# Nearby Surfaces Analysis Template
*For Cai Lab NanoMAP data analysis workflow. Created by: Ellie Lai. This version was edited on 2026/08/05, tested with **Python 3.13.5.***

**This script analyzes the localization of clusters of receptor within individual microvillus (MV) for all MV of the cell.**

This script processes 
1. results from python script **MV_and_Cluster_analysis** of this dataset, loads the output file ***df_data_loaded.xlsx***. with specified selections (by sheet_name) of MV and receptors clusters (e.g. over a defined distance above the cell surface with DT > 0.2). 
   * Note: the selection of MV/receptors were generated during data analysis in Imaris. The script **MV_and_Cluster_analysis** processes results exported from Imaris for ***MV selection***, ***Receptor1 selection***, and ***Receptor2 selection***.    
2. results from Imaris XTension **Nearby Surfaces**, loads the output file ***{dataset}_Surface1_Surface2_nearbySurface.csv.*** This file contains **ID information** for individual MV (Surface1) and receptor clusters(Surface2), and information on receptor clusters nearby a microvillus. For example: for *MV_id = i*, detected 3 nearby Receptor1 clusters with *Receptor1_cluster_id=[5,8,11]*.
   * Note: When running the Imaris XTension **Nearby Surfaces**, the users selects a Primary Surface (e.g. MV) and a Secondary Surface (e.g. Receptor1 or Receptor2) for process. The surfaces analyzed need to be from ***MV selection***, ***Receptor1 selection***, and ***Receptor2 selection***, to keep the consistent ID information of sub_surfaces within each group.
 
The template script uses **'Receptor1'** and **'Receptor2'** for receptors to be analyzed, which need to be replaced with the names of receptors to generate a **dataset-specitic script** for running.

**How to use:**
* Step 1 - Run the *NearbySurf_analysis_parameter.ipynb* to modify this template script to match the specific dataset you want to analyze. 
* Step 2 - Set the folder_path containing the data, and names of .csv files in the modified script and run the script. 

## 1. User Input:

In [ ]:
# Parameters (will be overridden by papermill)
folder_path = "/mypath/datafolder" # <-- Replace with path to the folder containing the input files here

# NearbySurfaces raw data
DIRECTORY = folder_path
Receptor1_CSV = "MV_Receptor1_NearBySurfaces.csv" # <-- Replace with the name of the MV_Receptor1_NearBySurfaces.csv file here
Receptor2_CSV = "MV_Receptor2_NearBySurfaces.csv" # <-- Replace with the name of the MV_Receptor2_NearBySurfaces.csv file here

# Files from MV analysis
SECONDARY_EXCEL_FILENAME = "df_data_loaded.xlsx" # <-- Replace with the name of the df_data_loaded.xlsx file here
# Define the sheet names for the Excel file
SECONDARY_SHEET_Receptor1 = "df_Receptor1_selec" # <-- Replace with the sheet name for Receptor1 Cluster Data here
SECONDARY_SHEET_Receptor2 = "df_Receptor2_selec" # <-- Replace with the sheet name for Receptor2 Cluster Data here
MV_SHEET_NAME = "df_MV_select"  # <-- Replace with the sheet name for MV Cluster Data here

## 2. Import Packages 

In [ ]:
import os
import pandas as pd
import numpy as np

## 3. File Loading and Preparation.

### 3.1. Load Receptor1 data

In [ ]:
# === FILE PATHS ===
csv_Receptor1_file = os.path.join(DIRECTORY, Receptor1_CSV)
csv_Receptor2_file = os.path.join(DIRECTORY, Receptor2_CSV)
secondary_file = os.path.join(DIRECTORY, SECONDARY_EXCEL_FILENAME)

# read NearbySurfaces Receptor1 CSV file and clean up file
if not os.path.isfile(csv_Receptor1_file):
    raise FileNotFoundError(f"CSV file not found: {csv_Receptor1_file}")

Receptor1_df = pd.read_csv(csv_Receptor1_file)

# Remove rows where Surf1 == 0
Receptor1_df_filtered = Receptor1_df[Receptor1_df['Surf1'] != 0]

# Set values close to 0.1 to exactly 0
threshold = 1e-6  

# Get column indices for Surf1 and Surf2
surf1_col = 0  # first column
surf2_col = 2  # third column

# Replace 0.1 values in Surf1 and Surf2 IDs with 0
Receptor1_df_filtered.iloc[:, surf1_col] = Receptor1_df_filtered.iloc[:, surf1_col].apply(lambda x: 0 if abs(x - 0.1) < threshold else x)
Receptor1_df_filtered.iloc[:, surf2_col] = Receptor1_df_filtered.iloc[:, surf2_col].apply(lambda x: 0 if abs(x - 0.1) < threshold else x)

# Inline output
print(Receptor1_df_filtered)

### 3.2. Receptor1 data clean up

In [ ]:
# make sure all Receptor1 IDs have valid rows in the MV Analysis Excel file
if not os.path.isfile(secondary_file):
    raise FileNotFoundError(f"Secondary Excel file not found: {secondary_file}")

df_data_Receptor1 = pd.read_excel(secondary_file, sheet_name=SECONDARY_SHEET_Receptor1)

# Extract valid IDs from Excel 
# Get the list of valid IDs from the df_data_loaded 'ID' column
valid_Receptor1_ids = df_data_Receptor1["ID"].astype(float).unique()  # Ensure numeric type for matching

# Filter the CSV based on Surf2 values
Receptor1_working_df = Receptor1_df_filtered[Receptor1_df_filtered["Surf2"].isin(valid_Receptor1_ids)]

print(Receptor1_working_df)

### 3.3. Load Receptor2 data

In [ ]:
# read NearbySurfaces Receptor2 CSV file and clean up file
if not os.path.isfile(csv_Receptor2_file):
    raise FileNotFoundError(f"CSV file not found: {csv_Receptor2_file}")

Receptor2_df = pd.read_csv(csv_Receptor2_file)

# Remove rows where Surf1 == 0
Receptor2_df_filtered = Receptor2_df[Receptor2_df['Surf1'] != 0]

# Set values close to 0.1 to exactly 0
threshold = 1e-6  # or use a larger value like 1e-4 if needed

# Get column indices for Surf1 and Surf2
surf1_col = 0  # first column
surf2_col = 2  # third column

# Replace 0.1 values in Surf1 and Surf2 IDs with 0
Receptor2_df_filtered.iloc[:, surf1_col] = Receptor2_df_filtered.iloc[:, surf1_col].apply(lambda x: 0 if abs(x - 0.1) < threshold else x)
Receptor2_df_filtered.iloc[:, surf2_col] = Receptor2_df_filtered.iloc[:, surf2_col].apply(lambda x: 0 if abs(x - 0.1) < threshold else x)

print(Receptor2_df_filtered)

### 3.4. Receptor2 data clean up

In [ ]:
# make sure all Receptor2 IDs have valid rows in the MV Analysis Excel file
if not os.path.isfile(secondary_file):
    raise FileNotFoundError(f"Secondary Excel file not found: {secondary_file}")

df_data_Receptor2 = pd.read_excel(secondary_file, sheet_name=SECONDARY_SHEET_Receptor2)

# Extract valid IDs from Excel 
# Get the list of valid IDs from the df_data_loaded 'ID' column
valid_Receptor2_ids = df_data_Receptor2["ID"].astype(float).unique()  # Ensure numeric type for matching

# Filter the CSV based on Surf2 values
Receptor2_working_df = Receptor2_df_filtered[Receptor2_df_filtered["Surf2"].isin(valid_Receptor2_ids)]

print(Receptor2_working_df)

## 4. MV Analysis

### 4.1. Global Receptor Count in MV

In [ ]:
# Filter for Surf2 "inside" MV
Receptor2_in = Receptor2_working_df[(Receptor2_working_df['DT_min'] <= 0) & (Receptor2_working_df['Surf2_Vox'] > 0)]
Receptor1_in = Receptor1_working_df[(Receptor1_working_df['DT_min'] <= 0) & (Receptor1_working_df['Surf2_Vox'] > 0)]

# Get sets of MV IDs with each receptor
Receptor2_mv = set(Receptor2_in['Surf1'].unique())
Receptor1_mv = set(Receptor1_in['Surf1'].unique())

# Get all unique MV IDs across both Receptor1 and Receptor2 (from filtered working df --> gets all MV IDs)
Receptor2_surf1_ids = set(Receptor2_working_df['Surf1'].unique())
Receptor1_surf1_ids = set(Receptor1_working_df['Surf1'].unique())

# Total set of all MV IDs seen in either file
all_mv = Receptor2_surf1_ids.union(Receptor1_surf1_ids)

# Total number of MV
total_mv = len(all_mv)

# Set operations
both = Receptor2_mv & Receptor1_mv
Receptor2_only = Receptor2_mv - Receptor1_mv
Receptor1_only = Receptor1_mv - Receptor2_mv
none = all_mv - (Receptor2_mv | Receptor1_mv)

# Build summary table
summary_df = pd.DataFrame({
    'Category': ['Total MV', 'Receptor2', 'Receptor1','Receptor2 only', 'Receptor1 only', 'Both', 'None'],
    'Count': [total_mv, len(Receptor2_mv), len(Receptor1_mv), len(Receptor2_only), len(Receptor1_only), len(both), len(none)],
})

# Add percentage column
summary_df['Percentage'] = (summary_df['Count'] / total_mv * 100).round(2)

# Show summary
print(summary_df)

### 4.2. Cluster Counts per MV

In [ ]:
# Classification table for each unique MV
classification = []

for mv_id in all_mv:
    if mv_id in Receptor2_mv and mv_id in Receptor1_mv:
        category = 'BOTH'
    elif mv_id in Receptor2_mv:
        category = 'Receptor2 ONLY'
    elif mv_id in Receptor1_mv:
        category = 'Receptor1 ONLY'
    else:
        category = 'NONE'
    classification.append({'Surf1': mv_id, 'Category': category})

# Convert to DataFrame
classification_df = pd.DataFrame(classification)
print(classification_df.head())

### 4.3. Cluster counts for each receptor

In [ ]:
# Count Receptor1 clusters per microvillus
Receptor1_cluster_counts = (
    Receptor1_in.groupby('Surf1')
    .size()
    .reset_index(name='Receptor1_cluster_count')
)

# Count Receptor2 clusters per microvillus
Receptor2_cluster_counts = (
    Receptor2_in.groupby('Surf1')
    .size()
    .reset_index(name='Receptor2_cluster_count')
)

# Merge the counts into one table
cluster_counts_df = pd.merge(
    Receptor2_cluster_counts,
    Receptor1_cluster_counts,
    on='Surf1',
    how='outer'  # includes all Surf1s from both Receptor2 and Receptor1
)

# Merge with classification table
merged_df = pd.merge(
    classification_df,
    cluster_counts_df,
    on='Surf1',
    how='left'  # keep all MV IDs from classification_df
)

# Fill missing cluster counts (e.g., where no Receptor2 or Receptor1 clusters were found in a particular MV) with 0
merged_df[['Receptor2_cluster_count', 'Receptor1_cluster_count']] = merged_df[
    ['Receptor2_cluster_count', 'Receptor1_cluster_count']
].fillna(0).astype(int)

# Add total cluster count column
merged_df['Total_clusters'] = merged_df['Receptor2_cluster_count'] + merged_df['Receptor1_cluster_count']

print(merged_df)

### 4.4. MV Clusters Info 
* grabs the data from MV Analysis df and finds the corresponding cluster information for all clusters that are in MV

In [ ]:
# Get IDs of Receptor1 and Receptor2 clusters in MV 
Receptor1_surf2_ids = (
    Receptor1_in.groupby('Surf1')['Surf2']
    .apply(list)
    .reset_index(name='Receptor1_Surf2_IDs')
)


Receptor2_surf2_ids = (
    Receptor2_in.groupby('Surf1')['Surf2']
    .apply(list)
    .reset_index(name='Receptor2_Surf2_IDs')
)

# Merge the lists
cluster_ids_df = pd.merge(Receptor2_surf2_ids, Receptor1_surf2_ids, on='Surf1', how='outer')

# Fill NaNs with empty lists
cluster_ids_df['Receptor2_Surf2_IDs'] = cluster_ids_df['Receptor2_Surf2_IDs'].apply(lambda x: x if isinstance(x, list) else [])
cluster_ids_df['Receptor1_Surf2_IDs'] = cluster_ids_df['Receptor1_Surf2_IDs'].apply(lambda x: x if isinstance(x, list) else [])

# Total cluster count
cluster_ids_df['Total_clusters'] = (
    cluster_ids_df['Receptor2_Surf2_IDs'].apply(len) +
    cluster_ids_df['Receptor1_Surf2_IDs'].apply(len)
)

# Final subset with MV that have at least 1 receptor cluster
final_cluster_df = cluster_ids_df[['Surf1', 'Total_clusters', 'Receptor2_Surf2_IDs', 'Receptor1_Surf2_IDs']]
final_cluster_df = final_cluster_df[final_cluster_df['Total_clusters'] > 0]

# Preview
print(final_cluster_df)

### 4.5. Receptors in MV Cluster Info

In [ ]:
# Step 1: Flatten Surf2 ID lists
Receptor2_ids = final_cluster_df['Receptor2_Surf2_IDs'].explode().dropna().unique().tolist()
Receptor1_ids = final_cluster_df['Receptor1_Surf2_IDs'].explode().dropna().unique().tolist()

# Step 2: Filter rows by matching ID
filtered_Receptor2_data = df_data_Receptor2[df_data_Receptor2['ID'].isin(Receptor2_ids)]
filtered_Receptor1_data = df_data_Receptor1[df_data_Receptor1['ID'].isin(Receptor1_ids)]

print(filtered_Receptor2_data)
print(filtered_Receptor1_data)

### 4.6. MV Cluster Info

In [ ]:
# Get list of MV IDs that have at least one cluster
MV_ids = final_cluster_df['Surf1'].unique().tolist()

# Read data
if not os.path.isfile(secondary_file):
    raise FileNotFoundError(f"Secondary Excel file not found: {secondary_file}")

df_data_MV = pd.read_excel(secondary_file, sheet_name=MV_SHEET_NAME)

# Filter the MV sheet by matching Surf1 IDs
filtered_MV_data = df_data_MV[df_data_MV['ID'].isin(MV_ids)]

print(filtered_MV_data)

### 4.7. MV with Both Receptors - Distance Ratio

In [ ]:
# List of microvilli with both Receptor1 and Receptor2 clusters
both_df = final_cluster_df[
    (final_cluster_df['Receptor2_Surf2_IDs'].apply(lambda x: len(x) > 0)) &
    (final_cluster_df['Receptor1_Surf2_IDs'].apply(lambda x: len(x) > 0))
].copy()

In [ ]:
# Receptor1 table --> Receptor1 IDs that are in MV with both Receptor1 and Receptor2
Receptor1_rows = []

for _, row in both_df.iterrows():
    mv_id = row['Surf1']
    mv_dist = df_data_MV.loc[df_data_MV['ID'] == mv_id, 'Distance to cell (max)'].values
    mv_dist = mv_dist[0] if len(mv_dist) > 0 else None

    for Receptor1_id in row['Receptor1_Surf2_IDs']:
        Receptor1_data = filtered_Receptor1_data[filtered_Receptor1_data['ID'] == Receptor1_id]
        if not Receptor1_data.empty:
            Receptor1_dist = Receptor1_data['Distance to cell (max)'].values[0]
            Receptor1_vol = Receptor1_data['Volume'].values[0]
            Receptor1_intensity = Receptor1_data['Intensity Sum Receptor1'].values[0]

            # --- add voxel overlaps ---
            Receptor1_overlap_mv = Receptor1_data['Voxel Overlap with MV'].values[0]
            Receptor1_overlap_Receptor1 = Receptor1_data['Voxel Overlap with Receptor1'].values[0]
            Receptor1_overlap_Receptor2 = Receptor1_data['Voxel Overlap with Receptor2'].values[0]

            # --- add overlap ratio ---
            Receptor1_overlap_ratio_mv = Receptor1_data['Overlap Volume Ratio to MV'].values[0]

        else:
            Receptor1_dist = Receptor1_vol = Receptor1_intensity = None
            Receptor1_overlap_mv = Receptor1_overlap_Receptor1 = Receptor1_overlap_Receptor2 = None

        Receptor1_rows.append({
            'Surf1': mv_id,
            'MV_Distance_to_Cell_Max': mv_dist,
            'Receptor1_ID': Receptor1_id,
            'Receptor1_Distance_to_Cell_Max': Receptor1_dist,
            'Receptor1_Volume': Receptor1_vol,
            'Receptor1_Intensity_Sum': Receptor1_intensity,
            'Receptor1_VoxelOverlap_MV': Receptor1_overlap_mv,
            'Receptor1_VoxelOverlap_Receptor1': Receptor1_overlap_Receptor1,
            'Receptor1_VoxelOverlap_Receptor2': Receptor1_overlap_Receptor2,
            'Receptor1_OverlapRatio_MV': Receptor1_overlap_ratio_mv
        })

Receptor1_cluster_df = pd.DataFrame(Receptor1_rows)

# Add Distance_Ratio column to Receptor1 table
Receptor1_cluster_df['Receptor1_Distance_Ratio'] = (
    Receptor1_cluster_df['Receptor1_Distance_to_Cell_Max'] / Receptor1_cluster_df['MV_Distance_to_Cell_Max']
)

print(Receptor1_cluster_df)

In [ ]:
# Receptor2 table --> Receptor1 IDs that are in MV with both Receptor1 and Receptor2
Receptor2_rows = []

for _, row in both_df.iterrows():
    mv_id = row['Surf1']
    mv_dist = df_data_MV.loc[df_data_MV['ID'] == mv_id, 'Distance to cell (max)'].values
    mv_dist = mv_dist[0] if len(mv_dist) > 0 else None

    for Receptor2_id in row['Receptor2_Surf2_IDs']:
        Receptor2_data = filtered_Receptor2_data[filtered_Receptor2_data['ID'] == Receptor2_id]
        if not Receptor2_data.empty:
            Receptor2_dist = Receptor2_data['Distance to cell (max)'].values[0]
            Receptor2_vol = Receptor2_data['Volume'].values[0]
            Receptor2_intensity = Receptor2_data['Intensity Sum Receptor2'].values[0]
        
            # --- add voxel overlaps ---
            Receptor2_overlap_mv = Receptor2_data['Voxel Overlap with MV'].values[0]
            Receptor2_overlap_Receptor1 = Receptor2_data['Voxel Overlap with Receptor1'].values[0]
            Receptor2_overlap_Receptor2 = Receptor2_data['Voxel Overlap with Receptor2'].values[0]
            
            # --- add overlap ratio ---
            Receptor2_overlap_ratio_mv = Receptor2_data['Overlap Volume Ratio to MV'].values[0]
        else:
            Receptor2_dist = Receptor2_vol = Receptor2_intensity = None
            Receptor2_overlap_mv = Receptor2_overlap_Receptor1 = Receptor2_overlap_Receptor2 = None

        Receptor2_rows.append({
            'Surf1': mv_id,
            'MV_Distance_to_Cell_Max': mv_dist,
            'Receptor2_ID': Receptor2_id,
            'Receptor2_Distance_to_Cell_Max': Receptor2_dist,
            'Receptor2_Volume': Receptor2_vol,
            'Receptor2_Intensity_Sum': Receptor2_intensity,
            'Receptor2_VoxelOverlap_MV': Receptor2_overlap_mv,
            'Receptor2_VoxelOverlap_Receptor1': Receptor2_overlap_Receptor1,
            'Receptor2_VoxelOverlap_Receptor2': Receptor2_overlap_Receptor2,
            'Receptor2_OverlapRatio_MV': Receptor2_overlap_ratio_mv
        })

Receptor2_cluster_df = pd.DataFrame(Receptor2_rows)

# Add Distance_Ratio column to Receptor2 table
Receptor2_cluster_df['Receptor2_Distance_Ratio'] = (
    Receptor2_cluster_df['Receptor2_Distance_to_Cell_Max'] / Receptor2_cluster_df['MV_Distance_to_Cell_Max']
)

print(Receptor2_cluster_df)

### 4.8. MV and Receptor Max Dist Ratio
Take one cluster per MV for each receptor (Max distance).

In [ ]:
# --- get max Receptor1 per MV ---
receptor1_max = (
    Receptor1_cluster_df.loc[
        Receptor1_cluster_df.groupby("Surf1")["Receptor1_Distance_to_Cell_Max"].idxmax()
    ]
    .copy()
    .rename(columns={
        "Receptor1_ID": "Max_Receptor1_ID",
        "Receptor1_Distance_to_Cell_Max": "Max_Receptor1_Distance"
    })
)

# --- get max Receptor2 per MV ---
receptor2_max = (
    Receptor2_cluster_df.loc[
        Receptor2_cluster_df.groupby("Surf1")["Receptor2_Distance_to_Cell_Max"].idxmax()
    ]
    .copy()
    .rename(columns={
        "Receptor2_ID": "Max_Receptor2_ID",
        "Receptor2_Distance_to_Cell_Max": "Max_Receptor2_Distance"
    })
)

# --- merge Receptor1 and Receptor2 on Surf1 ---
max_clusters = pd.merge(
    receptor1_max[["Surf1", "MV_Distance_to_Cell_Max", "Max_Receptor1_ID", "Max_Receptor1_Distance"]],
    receptor2_max[["Surf1", "MV_Distance_to_Cell_Max", "Max_Receptor2_ID", "Max_Receptor2_Distance"]],
    on="Surf1",
    suffixes=("_R1", "_R2")
)

# --- calculate ratios relative to MV distance ---
max_clusters["Receptor1_Distance_Ratio"] = (
    max_clusters["Max_Receptor1_Distance"] / max_clusters["MV_Distance_to_Cell_Max_R1"]
)
max_clusters["Receptor2_Distance_Ratio"] = (
    max_clusters["Max_Receptor2_Distance"] / max_clusters["MV_Distance_to_Cell_Max_R2"]
)

# --- calculate R1:R2 ratio ---
max_clusters["Receptor1_vs_Receptor2_Ratio"] = (
    max_clusters["Max_Receptor1_Distance"] / max_clusters["Max_Receptor2_Distance"]
)


### 4.9. Receptor Max Dist Ratio

In [ ]:
# --- Get max Receptor1 per MV ---
Receptor1_max = (
    Receptor1_cluster_df.loc[
        Receptor1_cluster_df.groupby('Surf1')['Receptor1_Distance_to_Cell_Max'].idxmax()
    ][['Surf1', 'Receptor1_ID', 'Receptor1_Distance_to_Cell_Max']]
    .reset_index(drop=True)
)

# --- Get max Receptor2 per MV ---
Receptor2_max = (
    Receptor2_cluster_df.loc[
        Receptor2_cluster_df.groupby('Surf1')['Receptor2_Distance_to_Cell_Max'].idxmax()
    ][['Surf1', 'Receptor2_ID', 'Receptor2_Distance_to_Cell_Max']]
    .reset_index(drop=True)
)

# --- Merge into one df ---
max_clusters_df = pd.merge(Receptor1_max, Receptor2_max, on='Surf1')

# Calculate ratio: Receptor1 / Receptor2
max_clusters_df['Receptor1_to_Receptor2_Distance_Ratio'] = (
    max_clusters_df['Receptor1_Distance_to_Cell_Max'] / 
    max_clusters_df['Receptor2_Distance_to_Cell_Max']
)

print(max_clusters_df)

### 4.10. Voxel Overlap of Both Receptors in Same MV

In [ ]:
## WARNING: will double count MV voxels if more than 1 cluster of same receptor in the MV 
## aka overlap fraction will be greater than 1

# Step 1: aggregate voxel overlaps per MV for Receptor1 and Receptor2
Receptor1_summary = (
    Receptor1_cluster_df.groupby("Surf1")["Receptor1_VoxelOverlap_MV"].sum()
    .reset_index()
    .rename(columns={"Receptor1_VoxelOverlap_MV": "Total_Receptor1_overlap_voxels"})
)

Receptor2_summary = (
    Receptor2_cluster_df.groupby("Surf1")["Receptor2_VoxelOverlap_MV"].sum()
    .reset_index()
    .rename(columns={"Receptor2_VoxelOverlap_MV": "Total_Receptor2_overlap_voxels"})
)

# Step 2: merge summaries together
overlap_summary = Receptor1_summary.merge(Receptor2_summary, on="Surf1", how="inner")

# Step 3: restrict to only MVs that have both receptors (from final_cluster_df)
both_ids = both_df["Surf1"].unique()
overlap_summary = overlap_summary[overlap_summary["Surf1"].isin(both_ids)]

# Step 4: merge with MV info sheet
overlap_summary = overlap_summary.merge(
    filtered_MV_data[["ID", "MV in MV"]],
    left_on="Surf1",
    right_on="ID",
    how="left"
)

# Step 5: compute fractions
overlap_summary["Receptor1_overlap_fraction"] = (
    overlap_summary["Total_Receptor1_overlap_voxels"] / overlap_summary["MV in MV"]
)
overlap_summary["Receptor2_overlap_fraction"] = (
    overlap_summary["Total_Receptor2_overlap_voxels"] / overlap_summary["MV in MV"]
)

# Optional combined fraction
overlap_summary["Combined_overlap_fraction"] = (
    (overlap_summary["Total_Receptor1_overlap_voxels"] +
     overlap_summary["Total_Receptor2_overlap_voxels"]) /
    overlap_summary["MV in MV"]
)

print(overlap_summary.head())

### 4.11. Overlap Estimate with Overlap Volume Ratio
* gives estimate of how much each MV is overlapped by Receptor1 and Receptor2 in MV with both receptors

In [ ]:
def receptor_union_verbose(cluster_df, ratio_col, receptor_label, both_df=None):
    """
    Compute overlap bounds and independence estimate per MV for a given receptor.
    
    Parameters
    ----------
    cluster_df : pd.DataFrame
        Per-cluster dataframe (e.g., CD8_cluster_df).
    ratio_col : str
        Name of the column with overlap ratio to MV (e.g., "CD8_OverlapRatio_MV").
    receptor_label : str
        Label for column prefixes (dependent on receptor name).
    both_df : pd.DataFrame, optional
        If provided, restricts to only MVs listed in both_df["Surf1"].
    
    Returns
    -------
    pd.DataFrame
        MV-level dataframe with bounds and estimates.
    """
    # Optional filter to only MVs with both receptors
    if both_df is not None:
        valid_ids = both_df["Surf1"].unique()
        cluster_df = cluster_df[cluster_df["Surf1"].isin(valid_ids)]

    rows = []
    for mv_id, group in cluster_df.groupby("Surf1"):
        rs = group[ratio_col].dropna().values
        if rs.size == 0:
            rows.append({
                "Surf1": mv_id,
                f"{receptor_label}_n": 0,
                f"{receptor_label}_sum_r": 0.0,
                f"{receptor_label}_max_r": 0.0,
                f"{receptor_label}_upper": 0.0,
                f"{receptor_label}_lower": 0.0,
                f"{receptor_label}_indep": 0.0
            })
            continue

        # Clip ratios to [0, 1]
        rs = np.clip(rs, 0.0, 1.0)

        sum_r = rs.sum()
        max_r = rs.max()
        upper = min(1.0, sum_r)
        lower = max_r

        # Stable independence estimate: 1 - exp(sum(log(1-r)))
        eps = 1e-12
        safe_rs = np.minimum(rs, 1 - eps)  # avoid log(0)
        logp = np.sum(np.log1p(-safe_rs))
        indep = 1 - np.exp(logp)

        rows.append({
            "Surf1": mv_id,
            f"{receptor_label}_n": int(rs.size),
            f"{receptor_label}_sum_r": sum_r,
            f"{receptor_label}_max_r": max_r,
            f"{receptor_label}_upper": upper,
            f"{receptor_label}_lower": lower,
            f"{receptor_label}_indep": indep
        })

    return pd.DataFrame(rows)

In [ ]:
# Compute per-MV bounds & estimates for each receptor
Receptor1_union_df = receptor_union_verbose(
    Receptor1_cluster_df, "Receptor1_OverlapRatio_MV", "Receptor1", both_df=both_df
)

Receptor2_union_df = receptor_union_verbose(
    Receptor2_cluster_df, "Receptor2_OverlapRatio_MV", "Receptor2", both_df=both_df
)

# Merge into one table
union_overlap_df = Receptor1_union_df.merge(Receptor2_union_df, on="Surf1", how="outer")

print(union_overlap_df.head())

## 5. Save results

In [ ]:
# Save files
# summary df (counts/percentages of MV with receptors)
# classification df (per-microvillus classification with receptors and cluster count for each receptor)
# final cluster df (tells specific cluster IDs found for each MV ID)
# filtered Receptor1 (cluster info from MV Analysis code for all Receptor1 clusters found in MV)
# filtered Receptor2 (cluster info from MV Analysis code for all Receptor2 clusters found in MV)
# filtered MV (MV cluster info from MV Analysis code for all MV with at least 1 receptor in it)
# Receptor1 cluster df (Receptor1 cluster info and distance ratio for all Receptor1 clusters in MV with both) 
# Receptor2 cluster df (Receptor2 cluster info and distance ratio for Receptor2 clusters in MV with both)

output_path = os.path.join(DIRECTORY, "MV_NearbySurf_analysis_results.xlsx")
with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    summary_df.to_excel(writer, sheet_name='Microvilli Occupancy Summary', index=False)
    merged_df.to_excel(writer, sheet_name='Per_Microvillus', index=False)
    final_cluster_df.to_excel(writer, sheet_name='MV Cluster IDs', index=False)
    filtered_Receptor1_data.to_excel(writer, sheet_name='Receptor1 ID Cluster Info', index=False)
    filtered_Receptor2_data.to_excel(writer, sheet_name='Receptor2 ID Cluster Info', index=False)
    filtered_MV_data.to_excel(writer,sheet_name='MV ID Cluster Info', index=False)
    Receptor1_cluster_df.to_excel(writer, sheet_name='Receptor1+Receptor2+ MV Receptor1 Dist Ratio', index=False)
    Receptor2_cluster_df.to_excel(writer, sheet_name='Receptor1+Receptor2+ MV Receptor2 Dist Ratio', index=False)
    max_clusters_df.to_excel(writer, sheet_name='MV and Receptor Max Dist Ratios', index=False)
    union_overlap_df.to_excel(writer, sheet_name='Overlap Info', index=False)